In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import unitary_group
from scipy.special import comb
from itertools import combinations

Purity

In [2]:
def density_matrix(psi):
    return np.matmul(psi,np.transpose(np.conjugate(psi)))

In [3]:
def partial_trace(psi, qubit_2_keep,n):
    
    rho = density_matrix(psi)
    num_qubit = n
    qubit_axis = [(i, num_qubit + i) for i in range(num_qubit)
                  if i not in qubit_2_keep]
    minus_factor = [(i, 2 * i) for i in range(len(qubit_axis))]
    minus_qubit_axis = [(q[0] - m[0], q[1] - m[1])
                        for q, m in zip(qubit_axis, minus_factor)]
    rho_res = np.reshape(rho, [2, 2] * num_qubit)
    qubit_left = num_qubit - len(qubit_axis)
    for i, j in minus_qubit_axis:
        rho_res = np.trace(rho_res, axis1=i, axis2=j)
    if qubit_left > 1:
        rho_res = np.reshape(rho_res, [2 ** qubit_left] * 2)

    return rho_res

In [5]:
def purity(psi, qubit_2_keep,n):
    rho_reduced = partial_trace(psi,qubit_2_keep,n)
    return np.trace(np.matmul(rho_reduced,rho_reduced))

Planar configurations

In [4]:
def p_config(n):
    system = [i for i in range(n)]
    if n%2 ==1:
        planar_subsystems = [[] for i in range(n)]
        for i in range(n):
            planar_subsystems[i] = list(np.sort([(i + k)%n for k in range(n//2)]))
    if n%2 ==0:
        planar_subsystems = [[] for i in range(n//2)]
        for i in range(n//2):
            planar_subsystems[i] = list(np.sort([(i + k)%n for k in range(n//2)]))
    return planar_subsystems

Absolute configurations

In [6]:
def a_config(n):
    system = [i for i in range(n)]
    m = comb(n, n//2, exact=True)
    if n%2 ==1:
        absolute_subsystems = [[] for i in range(m)]
        for i in range(m):
            absolute_subsystems[i] = list(combinations(system, n//2))[i]
            absolute_subsystems[i] = list(absolute_subsystems[i])
    if n%2 ==0:
        absolute_subsystems = [[] for i in range(m//2)]
        for i in range(m//2):
            absolute_subsystems[i] = list(combinations(system, n//2))[i]
            absolute_subsystems[i] = list(absolute_subsystems[i])
    return absolute_subsystems

Non-planar configurations

In [7]:
def n_config(n):
    a = a_config(n)
    p = p_config(n)
    np = [x for x in a if x not in p]
    return np

Sampling from states and calculating their purities

In [8]:
n = 10
N = 2**n
m = comb(n, n//2, exact=True)
planar_configuration = p_config(n)
npc = len(planar_configuration)
non_planar_configuration = n_config(n)
nnpc = len(non_planar_configuration)
n_samples = 40000
psi_0 = np.zeros((N,1),dtype=complex)
psi_0[0] = 1
samples = np.zeros((n_samples,N,1),dtype=complex)
purity_samples_p = np.zeros(n_samples)
purity_samples_a = np.zeros(n_samples)

from datetime import datetime
start=datetime.now()


for i in range(n_samples):
    samples[i] = np.matmul(unitary_group.rvs(N),psi_0)
    for j in range(npc):
        purity_samples_p[i] += purity(samples[i],planar_configuration[j],n)
    purity_samples_a[i] = purity_samples_p[i]
    purity_samples_p[i] /= npc
    for k in range(nnpc):
        purity_samples_a[i] += purity(samples[i],non_planar_configuration[k],n)
    purity_samples_a[i] /= (npc + nnpc)
    
print (datetime.now()-start)

C:\Users\Kourosh\AppData\Local\Temp\ipykernel_2724\689745457.py:22: ComplexWarning: Casting complex values to real discards the imaginary part
  purity_samples_p[i] += purity(samples[i],planar_configuration[j],n)
C:\Users\Kourosh\AppData\Local\Temp\ipykernel_2724\689745457.py:26: ComplexWarning: Casting complex values to real discards the imaginary part
  purity_samples_a[i] += purity(samples[i],non_planar_configuration[k],n)


3:05:37.636106


In [24]:
y,x,patches = plt.hist(purity_samples_4,bins = 100,density=True,histtype='step',label='n=4')
y,x,patches = plt.hist(purity_samples_5,bins = 100,density=True,histtype='step',label='n=5')
y,x,patches = plt.hist(purity_samples_6,bins = 100,density=True,histtype='step',label='n=6')
y,x,patches = plt.hist(purity_samples_7,bins = 100,density=True,histtype='step',label='n=7')
y,x,patches = plt.hist(purity_samples_8,bins = 100,density=True,histtype='step',label='n=8')
font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 22}
plt.rc('font', **font)
plt.rc('xtick', labelsize=20) 
plt.rc('ytick', labelsize=20) 
plt.legend(loc='best')
plt.show()




findfont: Font family ['normal'] not found. Falling back to DejaVu Sans.


In [9]:
np.savetxt('purity_samples_a(n=10)',purity_samples_a)
np.savetxt('purity_samples_p(n=10)',purity_samples_p)

In [3]:
purity_samples_10 = np.loadtxt('purity_samples_p(n=10)')
purity_samples_a = np.loadtxt('purity_samples_a(n=10)')

Mean and Variance

In [16]:
y,x,patches = plt.hist(purity_samples_10,bins = 100,density=True,histtype='step',label='n=8')
S_p = np.var(purity_samples_10)
S_a = np.var(purity_samples_a)
M_p = np.mean(purity_samples_10)
M_a = np.mean(purity_samples_a)

x=np.delete(x,0)
def Gauss(x, x0,a, sigma):
    return a * np.exp(-(x - x0)**2 / (2 * sigma))

from scipy.optimize import curve_fit
popt,pcov = curve_fit(Gauss, x, y)

font = {'size'   : 12}
plt.rc('font', **font)
plt.rc('xtick', labelsize=20) 
plt.rc('ytick', labelsize=20) 

plt.plot(x,y,'o',label='Data',color='green')
plt.plot(x, Gauss(x, *popt), 'r-', label='Gaussian fit, Sigma='+str('%.5f'%(np.sqrt(S_p)))+'Mean='+str('%.5f'%M_p))
plt.legend()
plt.show()


RuntimeError: Optimal parameters not found: Number of calls to function has reached maxfev = 800.

In [4]:
plt.hist(purity_samples_10,bins = 100,density=True,histtype='step',label='Planar')
plt.hist(purity_samples_a,bins = 100,density=True,histtype='step',label='Absolute')
plt.legend()
plt.show()

An example for PMES

In [ ]:
phi = np.random.rand()*2*np.pi
alpha = np.random.rand()*2*np.pi
beta = np.random.rand()*2*np.pi
gamma = np.random.rand()*2*np.pi
delta = np.random.rand()*2*np.pi

basis = []
for i in range(16):
    basis += [np.zeros(16)]
    basis[i][i] = 1
    basis[i] = basis[i].reshape(16,1)
psi_B = 0.5*np.cos(phi)*(np.exp(1j*alpha) * basis[0] + np.exp(-1j*alpha) * basis[15] + np.exp(1j*beta) * basis[5] + np.exp(-1j*beta) * basis[10]) + 0.5*np.sin(phi)*(np.exp(1j*gamma) * basis[3] - np.exp(-1j*gamma) * basis[12] + np.exp(1j*delta) * basis[6] - np.exp(-1j*delta) * basis[9])

#psi_B = np.array([[0.5*np.cos(phi)*(np.exp(1j*alpha))],[0],[0],[0.5*np.sin(phi)*(np.exp(1j*gamma))],[0],[0.5*np.cos(phi)*(np.exp(1j*beta))],[0.5*np.sin(phi)*(np.exp(1j*delta))],[0],[0],[0.5*np.sin(phi)*(np.exp(-1j*delta))],[0.5*np.cos(phi)*(np.exp(-1j*beta))],[0],[0.5*np.sin(phi)*(np.exp(-1j*gamma))],[0],[0],[0.5*np.cos(phi)*(np.exp(-1j*alpha))]])
#np.transpose(np.conjugate(psi_B))
purity(psi_B,[3,0],4)

W state

In [ ]:
n = 5
N = 2**n
W = np.zeros((N,1), dtype=complex)
up_qubit = np.zeros((2,1), dtype=complex)
up_qubit[0] = 1
down_qubit = np.zeros((2,1), dtype=complex)
down_qubit[1] = 1
qubit = np.array([up_qubit,down_qubit])
for i in range(n):
    l = np.zeros(n, dtype=int)
    l[i] = 1
    for j in range(n):
        if j == 0:
            state = qubit[l[j]]
        else:
            state = np.kron(state,qubit[l[j]])
    W += state
W /= np.sqrt(n)      

In [ ]:
WAP = 0
WPP = 0
planar_configuration = p_config(n)
npc = len(planar_configuration)
non_planar_configuration = n_config(n)
nnpc = len(non_planar_configuration)
for j in range(npc):
    WPP += purity(W,planar_configuration[j],n)
WAP = WPP
WPP /= npc
for k in range(nnpc):
    WAP += purity(W,non_planar_configuration[k],n)
WAP /= (npc + nnpc)

In [81]:
'%.3f'%(1324343032.324325235)

'1324343032.324'

In [48]:
import qutip as qt
psi=np.matmul(unitary_group.rvs(N),psi_0)
psi1 = qt.Qobj(np.matmul(psi,np.transpose(np.conjugate(psi))),dims=[[2,1],[2,1]])


ValueError: Qobj has smaller dims [[2, 1], [2, 1]] than underlying shape (4, 4)

In [44]:
qt.ptrace(psi1,0)

Quantum object: dims = [[4], [4]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[ 0.47718899+0.j         -0.14301787+0.15661355j  0.14176964+0.25224277j
  -0.13790684-0.31898965j]
 [-0.14301787-0.15661355j  0.09426436+0.j          0.0402965 -0.12212828j
  -0.06336056+0.14086516j]
 [ 0.14176964-0.25224277j  0.0402965 +0.12212828j  0.17545469+0.j
  -0.20958957-0.02187192j]
 [-0.13790684+0.31898965j -0.06336056-0.14086516j -0.20958957+0.02187192j
   0.25309195+0.j        ]]

In [25]:
qt.tensor(qt.basis(2, 0), qt.basis(2, 1))

Quantum object: dims = [[2, 2], [1, 1]], shape = (4, 1), type = ket
Qobj data =
[[0.]
 [1.]
 [0.]
 [0.]]

In [43]:
psi1

Quantum object: dims = [[4], [4]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[ 0.47718899+0.j         -0.14301787+0.15661355j  0.14176964+0.25224277j
  -0.13790684-0.31898965j]
 [-0.14301787-0.15661355j  0.09426436+0.j          0.0402965 -0.12212828j
  -0.06336056+0.14086516j]
 [ 0.14176964-0.25224277j  0.0402965 +0.12212828j  0.17545469+0.j
  -0.20958957-0.02187192j]
 [-0.13790684+0.31898965j -0.06336056-0.14086516j -0.20958957+0.02187192j
   0.25309195+0.j        ]]